In [3]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/cleaned_projects.csv")
print(df_clean.shape)
df_clean.head()

(3000, 17)


,project_id,project_name,project_type,state,district,land_area_hectares,affected_families,compensation_disbursed_pct,active_legal_disputes,possession_pct,rnr_progress_pct,stakeholder_responsiveness,dept_historical_delay_rate,days_since_notification,planned_completion_days,is_delayed,delay_days
0,LA-OD-00000,Irrigation Canal Project 0,Irrigation Canal,Odisha,Cuttack,55.4,295,52.7,1,49.1,31.1,0.51,0.67,338,432,0,0
1,LA-GU-00001,Irrigation Canal Project 1,Irrigation Canal,Gujarat,Vadodara,73.5,156,28.3,0,68.2,33.9,0.56,0.67,240,230,1,1245
2,LA-GU-00002,Irrigation Canal Project 2,Irrigation Canal,Gujarat,Vadodara,109.4,113,24.4,1,43.7,4.9,0.12,0.63,134,285,1,651
3,LA-KA-00003,Irrigation Canal Project 3,Irrigation Canal,Karnataka,Belagavi,60.5,113,60.3,1,23.2,3.0,0.41,0.42,141,406,0,0
4,LA-KA-00004,Rail Corridor Project 4,Rail Corridor,Karnataka,Bengaluru Rural,47.3,84,31.3,4,35.5,17.2,0.20,0.68,95,488,1,1203


In [4]:
# --- compensation_pending_ratio: seedha compensation_disbursed_pct ka opposite
df_clean["compensation_pending_ratio"] = 100 - df_clean["compensation_disbursed_pct"]

# --- legal_dispute_density: disputes ko project size (families) se normalize karna
df_clean["legal_dispute_density"] = df_clean["active_legal_disputes"] / (df_clean["affected_families"] / 100)

# --- rnr_gap: possession aur R&R progress ke beech ka gap (sirf "risky direction" — 
# jab possession R&R se aage ho — count karo, negative gap ko 0 kar do)
df_clean["rnr_gap"] = (df_clean["possession_pct"] - df_clean["rnr_progress_pct"]).clip(lower=0)

# --- stage_velocity: kitni tezi se possession badh raha hai (per day)
df_clean["stage_velocity"] = df_clean["possession_pct"] / df_clean["days_since_notification"].replace(0, 1)

# --- family_density: kitni families per hectare (zameen ka congestion)
df_clean["family_density"] = df_clean["affected_families"] / df_clean["land_area_hectares"]

print("New columns added:", df_clean.shape)

New columns added: (3000, 22)


In [5]:
new_feats = ["compensation_pending_ratio", "legal_dispute_density", "rnr_gap",
             "stage_velocity", "family_density"]
print(df_clean[new_feats + ["is_delayed"]].corr()["is_delayed"].sort_values())

family_density               -0.004178
stage_velocity                0.001354
rnr_gap                       0.097263
legal_dispute_density         0.269765
compensation_pending_ratio    0.394274
is_delayed                    1.000000
Name: is_delayed, dtype: float64


In [6]:
df_clean.to_csv("../data/processed/featured_projects.csv", index=False)
print("Saved: data/processed/featured_projects.csv  shape:", df_clean.shape)

Saved: data/processed/featured_projects.csv  shape: (3000, 22)


In [7]:
df_clean.head(1)

,project_id,project_name,project_type,state,district,land_area_hectares,affected_families,compensation_disbursed_pct,active_legal_disputes,possession_pct,...,dept_historical_delay_rate,days_since_notification,planned_completion_days,is_delayed,delay_days,compensation_pending_ratio,legal_dispute_density,rnr_gap,stage_velocity,family_density
0,LA-OD-00000,Irrigation Canal Project 0,Irrigation Canal,Odisha,Cuttack,55.4,295,52.7,1,49.1,...,0.67,338,432,0,0,47.3,0.338983,18.0,0.145266,5.32491
